In [2]:
import os
import sys

# Decimos al notebook que mire la raíz para encontrar agent.py
if '..' not in sys.path:
    sys.path.append('..')

import pandas as pd
from sqlalchemy import create_engine
# Importamos la función real del archivo agent.py de la raíz
from agent import pregunta_a_sql

def agente_responder(pregunta_usuario):
    try:
        # 1. Llamamos a la función importada (pregunta_a_sql)
        sql = pregunta_a_sql(pregunta_usuario)
        print("❓ Pregunta del usuario:", pregunta_usuario)
        print("💻 SQL generado por el Agente:\n", sql)
        
        # 2. Tu motor de conexión de SQLAlchemy
        usuario = os.getenv("DB_USER")
        password = os.getenv("DB_PASSWORD")
        host = os.getenv("DB_HOST", "localhost")
        port = os.getenv("DB_PORT", "3306")
        db_name = os.getenv("DB_NAME")
        
        engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}:{port}/{db_name}")
        
        # 3. Ejecución y volcado a DataFrame
        df = pd.read_sql(sql, engine)
        
        if not df.empty:
            print("📊 Resultados obtenidos de la base de datos:")
            return df
        else:
            print("⚠️ La consulta se ejecutó bien, pero no devolvió ningún registro.")
            return None
            
    except Exception as error:
        print(f"❌ Error al procesar la solicitud: {error}")
        return None

In [3]:
import os
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# 1. Cargamos las variables de entorno de forma segura
load_dotenv()

try:
    print(f"Intentando conectar a la base de datos de forma segura...")
    
    # 2. Conectamos usando os.getenv() para no mostrar contraseñas en el código
    conexion = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        connection_timeout=5
    )
    print("¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉\n")
    
    # 3. Validamos las tablas existentes
    cursor = conexion.cursor()
    cursor.execute("SHOW TABLES;")
    tablas = cursor.fetchall()
    
    print(f"Tablas sincronizadas desde '{os.getenv('DB_NAME')}':")
    for t in tablas:
        print(f"  • {t[0]}")
        
    conexion.close()

except mysql.connector.Error as error:
    print(f"\nError de conexión: {error}")
    print("Chequeá que el archivo .env esté bien guardado y en la misma carpeta.")

Intentando conectar a la base de datos de forma segura...
¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉

Tablas sincronizadas desde 'final':
  • auditoria_prestamos
  • autor
  • ejemplar
  • genero
  • libro
  • libro_autor
  • libro_genero
  • prestamo
  • sancion
  • socio
  • tipo_sancion


In [4]:
def agente_responder(pregunta_usuario):
    try:
        # 1. Llamamos a la función real que importamos desde agent.py
        sql = pregunta_a_sql(pregunta_usuario)
        print("❓ Pregunta del usuario:", pregunta_usuario)
        print("💻 SQL generado por el Agente:\n", sql)
        
        # 2. Conexión limpia con SQLAlchemy
        usuario = os.getenv("DB_USER")
        password = os.getenv("DB_PASSWORD")
        host = os.getenv("DB_HOST", "localhost")
        port = os.getenv("DB_PORT", "3306")
        db_name = os.getenv("DB_NAME")
        
        engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}:{port}/{db_name}")
        
        # 3. Ejecutamos y traemos los datos
        df = pd.read_sql(sql, engine)
        return df
            
    except Exception as error:
        print(f"❌ Error al procesar la solicitud: {error}")
        return None

In [5]:
import os
import pandas as pd
from sqlalchemy import create_engine  # 💡 SUMAMOS ESTA LÍNEA

def agente_responder(pregunta_usuario):
    try:
        # 1. El agente genera el SQL usando el archivo de la raíz
        sql = pregunta_a_sql(pregunta_usuario)
        print("❓ Pregunta del usuario:", pregunta_usuario)
        print("💻 SQL generado por el Agente:\n", sql)
        
        # 2. Leemos las variables del entorno
        usuario = os.getenv("DB_USER")
        password = os.getenv("DB_PASSWORD")
        host = os.getenv("DB_HOST", "localhost")
        port = os.getenv("DB_PORT", "3306")
        db_name = os.getenv("DB_NAME")
        
        # 💡 CAMBIO CLAVE: Usamos create_engine para eliminar el UserWarning
        engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}:{port}/{db_name}")
        
        # 3. Le pasamos el 'engine' a Pandas en vez de la conexión vieja
        df = pd.read_sql(sql, engine)
        
        if not df.empty:
            print("\n📊 Resultados obtenidos de la base de datos:")
            return df
        else:
            print("\n⚠️ La consulta se ejecutó bien, pero no devolvió ningún registro.")
            return None
            
    except Exception as error:
        print(f"❌ Error al procesar la solicitud: {error}")
        return None

In [6]:
agente_responder("Mostrame los títulos de los libros escritos por el autor Isaac Asimov")

❓ Pregunta del usuario: Mostrame los títulos de los libros escritos por el autor Isaac Asimov
💻 SQL generado por el Agente:
 SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'Isaac' AND a.apellido = 'Asimov'

📊 Resultados obtenidos de la base de datos:


,titulo
0,Fundación
1,Fundación e Imperio


In [7]:
agente_responder("¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos")

❓ Pregunta del usuario: ¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos
💻 SQL generado por el Agente:
 SELECT DISTINCT s.nombre, s.apellido FROM socio s JOIN prestamo p ON s.id_socio = p.id_socio WHERE p.estado = 'VENCIDO'

📊 Resultados obtenidos de la base de datos:


,nombre,apellido
0,Juan,Rodríguez
1,Valentina,Álvarez
2,Mateo,Romero
3,Emma,Sánchez


In [8]:
# ====================================================================
# DEMOSTRACIÓN FINAL DEL AGENTE (PRUEBAS OBLIGATORIAS DEL TP)
# ====================================================================

print("--- PRUEBA 1: Búsqueda con Joins y agregación ---")
display(agente_responder("¿Qué libros escribió el autor J.R.R. Tolkien?"))
print("-" * 60)

print("\n--- PRUEBA 2: Filtros de estados y fechas ---")
display(agente_responder("Mostrame los socios que están suspendidos"))
print("-" * 60)

print("\n--- PRUEBA 3: Funciones de agregación y ordenamiento ---")
display(agente_responder("¿Cuáles son los 3 libros con mayor stock total en la biblioteca?"))
print("-" * 60)

print("\n--- PRUEBA 4: Cruce complejo de tres tablas ---")
display(agente_responder("Listá los títulos de los libros que pertenecen al género Fantasía"))
print("-" * 60)

--- PRUEBA 1: Búsqueda con Joins y agregación ---


❓ Pregunta del usuario: ¿Qué libros escribió el autor J.R.R. Tolkien?
💻 SQL generado por el Agente:
 SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'J.R.R.' AND a.apellido = 'Tolkien'

📊 Resultados obtenidos de la base de datos:


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey


------------------------------------------------------------

--- PRUEBA 2: Filtros de estados y fechas ---
❓ Pregunta del usuario: Mostrame los socios que están suspendidos
💻 SQL generado por el Agente:
 SELECT id_socio, dni, nombre, apellido FROM socio WHERE estado = 'SUSPENDIDO'

📊 Resultados obtenidos de la base de datos:


,id_socio,dni,nombre,apellido
0,3,37333444,Juan,Rodríguez


------------------------------------------------------------

--- PRUEBA 3: Funciones de agregación y ordenamiento ---
❓ Pregunta del usuario: ¿Cuáles son los 3 libros con mayor stock total en la biblioteca?
💻 SQL generado por el Agente:
 SELECT titulo, stock_total FROM libro ORDER BY stock_total DESC LIMIT 3

📊 Resultados obtenidos de la base de datos:


,titulo,stock_total
0,1984,5
1,El Señor de los Anillos: La Comunidad del Anillo,5
2,Sapiens: De animales a dioses,4


------------------------------------------------------------

--- PRUEBA 4: Cruce complejo de tres tablas ---
❓ Pregunta del usuario: Listá los títulos de los libros que pertenecen al género Fantasía
💻 SQL generado por el Agente:
 SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_genero lg ON l.isbn = lg.isbn 
JOIN genero g ON lg.id_genero = g.id_genero 
WHERE g.nombre = 'Fantasía'

📊 Resultados obtenidos de la base de datos:


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey
3,Juego de Tronos
4,Choque de Reyes


------------------------------------------------------------


In [9]:
agente_responder("¿Cuántos libros hay de cada género en la biblioteca?")

❓ Pregunta del usuario: ¿Cuántos libros hay de cada género en la biblioteca?
💻 SQL generado por el Agente:
 SELECT g.nombre, COUNT(l.isbn) 
FROM genero g 
JOIN libro_genero lg ON g.id_genero = lg.id_genero 
JOIN libro l ON lg.isbn = l.isbn 
GROUP BY g.nombre

📊 Resultados obtenidos de la base de datos:


,nombre,COUNT(l.isbn)
0,Ciencia Ficción,5
1,Divulgación Científica,4
2,Fantasía,5
3,Novela Histórica,3
4,Terror,2


In [10]:
agente_responder("Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto.")

❓ Pregunta del usuario: Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto.
💻 SQL generado por el Agente:
 SELECT DISTINCT e1.isbn, l.titulo 
FROM prestamo p1 
JOIN ejemplar e1 ON p1.id_ejemplar = e1.id_ejemplar 
JOIN libro l ON e1.isbn = l.isbn 
JOIN prestamo p2 ON p1.id_ejemplar <> p2.id_ejemplar 
JOIN ejemplar e2 ON p2.id_ejemplar = e2.id_ejemplar 
WHERE e1.isbn = e2.isbn 
AND p1.fecha_prestamo <= p2.fecha_devolucion 
AND p1.fecha_devolucion >= p2.fecha_prestamo 
AND p1.fecha_devolucion IS NOT NULL 
AND p2.fecha_devolucion IS NOT NULL;

📊 Resultados obtenidos de la base de datos:


,isbn,titulo
0,9780261103573,El Señor de los Anillos: La Comunidad del Anillo
1,9780451457998,2001: Una Odisea Espacial
2,9780553103540,Juego de Tronos
3,9780553293357,Fundación


In [13]:
agente_responder("¿Qué libros están por agotarse?")

❓ Pregunta del usuario: ¿Qué libros están por agotarse?
💻 SQL generado por el Agente:
 SELECT isbn, titulo, stock_total, stock_disponible FROM vista_stock_critico

📊 Resultados obtenidos de la base de datos:


,isbn,titulo,stock_total,stock_disponible
0,9780307743657,El Resplandor,3,2
1,9780345373595,Cosmos,2,2
2,9780345539434,Un punto azul pálido,2,2
3,9780451457998,2001: Una Odisea Espacial,3,2
4,9780553103540,Juego de Tronos,4,2
5,9780553106633,Choque de Reyes,2,2
6,9780553294385,Fundación e Imperio,2,2
7,9781501142970,It (Eso),3,2


In [15]:
agente_responder("Mostrame los socios morosos")

❓ Pregunta del usuario: Mostrame los socios morosos
💻 SQL generado por el Agente:
 SELECT * FROM vista_socios_morosos

📊 Resultados obtenidos de la base de datos:


,id_socio,dni,nombre,apellido,id_prestamo,fecha_vencimiento
0,3,37333444,Juan,Rodríguez,47,2026-05-16
1,8,40333003,Valentina,Álvarez,48,2026-05-17
2,9,40444004,Mateo,Romero,49,2026-05-18
3,10,40555005,Emma,Sánchez,50,2026-05-20
